<a href="https://colab.research.google.com/github/Dani2003/paper-implementations/blob/main/MoE_Phase1-3_Core_Architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mixture of Experts (MoE) __ Phase 1: Core Mathematical Architecture

### Objective
We are building a clean-room implementation of a Sparsely-Gated Mixture of Experts (MoE) layer from scratch in PyTorch. The goal of Phase 1 is to implement and verify the core routing mechanics while introducing a system-level guard against hardware under-utilization.

### Core Architecture Components
1. **Noisy Top-K Router:** Implements Shazeer et al. style gating with learnable noise to assist with expert exploration.
2. **Dynamic Dispatch Loop:** Isolates token hidden states and routes them exclusively to their selected expert networks.
3. **Auxiliary Load-Balancing Loss ($\mathcal{L}_{bal}$):** A mathematical penalty function that forces the router to distribute tokens uniformly across all available hardware, preventing **Router Collapse** (where one expert processes 100% of the workload while the others sit idle).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

class FFNExpert(nn.Module):
    """A standard Feed-Forward Network acting as a single expert."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.act = nn.GELU()

    def forward(self, x):
        return self.w_2(self.act(self.w_1(x)))

class NoisyTopKGatedRouter(nn.Module):
    """
    Shazeer et al. style Noisy Top-K Router.
    Introduces learnable noise to assist with expert exploration and load balancing.
    """
    def __init__(self, d_model, num_experts, k=2):
        super().__init__()
        self.num_experts = num_experts
        self.k = k
        self.w_gate = nn.Linear(d_model, num_experts, bias=False)
        self.w_noise = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x, training=True):
        logits = self.w_gate(x)
        if training:
            noise_scale = F.softplus(self.w_noise(x))
            noise = torch.randn_like(logits) * noise_scale
            logits = logits + noise

        topk_gates, topk_indices = torch.topk(logits, self.k, dim=-1)
        softmax_gates = F.softmax(topk_gates, dim=-1)
        return softmax_gates, topk_indices, logits

class ProductionMoELayer(nn.Module):
    def __init__(self, num_experts=8, k=2, d_model=512, d_ff=2048):
        super().__init__()
        self.num_experts = num_experts
        self.k = k
        self.router = NoisyTopKGatedRouter(d_model, num_experts, k)
        self.experts = nn.ModuleList([FFNExpert(d_model, d_ff) for _ in range(num_experts)])

    def _compute_auxiliary_loss(self, router_logits, topk_indices):
        """Measures and rewards switch routing diversity across the token batch."""
        total_tokens = router_logits.size(0)
        gates_all = F.softmax(router_logits, dim=-1)
        p_i = gates_all.mean(dim=0)

        expert_mask = torch.zeros(total_tokens, self.num_experts, device=router_logits.device)
        expert_mask.scatter_(1, topk_indices, 1.0)
        f_i = expert_mask.mean(dim=0)

        return self.num_experts * torch.sum(p_i * f_i)

    def forward(self, x):
        orig_shape = x.shape
        x_flat = x.view(-1, orig_shape[-1])
        gates, indices, raw_logits = self.router(x_flat, training=self.training)
        aux_loss = self._compute_auxiliary_loss(raw_logits, indices)

        output_buffer = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.experts):
            token_mask = (indices == i).any(dim=-1)
            if token_mask.any():
                token_indices = torch.where(token_mask)[0]
                gate_positions = (indices[token_indices] == i).nonzero(as_tuple=True)[1]
                expert_gates = gates[token_indices, gate_positions].unsqueeze(-1)

                output_buffer[token_indices] += expert(x_flat[token_indices]) * expert_gates

        return output_buffer.view(orig_shape), aux_loss

In [ ]:
# Set up a clean replication environment
torch.manual_seed(42)

# Dimensions: Batch=32, Sequence Length=512, Embedding Dimension=512
B, S, D = 32, 512, 512
simulated_tokens = torch.randn(B, S, D)

# Instantiate our 8-expert architecture selecting Top-2 per token
moe_layer = ProductionMoELayer(num_experts=8, k=2, d_model=D, d_ff=2048)

print("--- RUNNING PHASE 1 EXPERIMENTATION ---")
start_time = time.perf_counter()
output, aux_loss = moe_layer(simulated_tokens)
elapsed_ms = (time.perf_counter() - start_time) * 1000

print(f"Forward Pass Duration: {elapsed_ms:.2f} ms")
print(f"Output Matrix Dimensions: {list(output.shape)}")
print(f"Calculated Auxiliary Balance Loss: {aux_loss.item():.4f}")

# Target metrics interpretation
print("\nVerification Rules:")
if aux_loss.item() >= 1.0:
    print("System Stable: Gating distribution is healthy and exploring all experts.")
else:
    print("Risk of Router Collapse: Gating nodes are bottlenecking on limited experts.")

# Phase 2: Capacity Factors and Token Dropping

### Objective
In a production framework, we cannot allow an individual expert's queue to grow indefinitely. If one expert bottlenecks, the entire parallel execution grid stalls. We are implementing a structural capacity limit.

### Key Optimization Mechanics
1. **Expert Capacity Calculation:** We define a capacity threshold based on a multiplier called the `capacity_factor`. The maximum tokens allowed per expert slot is calculated via:
   $$\text{Expert Capacity} = \left( \frac{\text{Total Tokens} \times K}{\text{Number of Experts}} \right) \times \text{Capacity Factor}$$
2. **Token Dropping Loop:** Tokens that are routed to an expert after it hits its maximum capacity ceiling are safely dropped (or passed straight through via a residual connection), preserving fixed tensor shapes and saving valuable GPU clock cycles.

In [ ]:
class ProductionMoELayerWithCapacity(nn.Module):
    def __init__(self, num_experts=8, k=2, d_model=512, d_ff=2048, capacity_factor=1.0):
        super().__init__()
        self.num_experts = num_experts
        self.k = k
        self.capacity_factor = capacity_factor

        self.router = NoisyTopKGatedRouter(d_model, num_experts, k)
        self.experts = nn.ModuleList([FFNExpert(d_model, d_ff) for _ in range(num_experts)])

    def forward(self, x):
        orig_shape = x.shape
        x_flat = x.view(-1, orig_shape[-1])
        total_tokens = x_flat.size(0)

        # Calculate exact capacity limit per expert
        expert_capacity = int((total_tokens * self.k / self.num_experts) * self.capacity_factor)

        # 1. Gather routing assignments
        gates, indices, raw_logits = self.router(x_flat, training=self.training)

        output_buffer = torch.zeros_like(x_flat)
        total_dropped = 0

        # 2. Optimized Dispatch Loop with Capacity Constraints
        for i, expert in enumerate(self.experts):
            # Find tokens routing to this expert
            token_mask = (indices == i).any(dim=-1)

            if token_mask.any():
                token_indices = torch.where(token_mask)[0]

                # Enforce capacity constraint limits
                if len(token_indices) > expert_capacity:
                    total_dropped += (len(token_indices) - expert_capacity)
                    token_indices = token_indices[:expert_capacity]

                if len(token_indices) > 0:
                    # Match selected tokens to their gating weights
                    gate_positions = (indices[token_indices] == i).nonzero(as_tuple=True)[1]
                    expert_gates = gates[token_indices, gate_positions].unsqueeze(-1)

                    # Compute forward pass exclusively for tokens within the capacity window
                    output_buffer[token_indices] += expert(x_flat[token_indices]) * expert_gates

        dropped_percentage = (total_dropped / (total_tokens * self.k)) * 100
        return output_buffer.view(orig_shape), dropped_percentage

In [ ]:
# Instantiate the optimized layer with a strict capacity factor of 1.0
optimized_moe = ProductionMoELayerWithCapacity(num_experts=8, k=2, d_model=D, d_ff=2048, capacity_factor=1.0)

print("--- RUNNING PHASE 2 CONSTRAINED EXPERIMENTATION ---")
start_time = time.perf_counter()
opt_output, dropped_pct = optimized_moe(simulated_tokens)
opt_elapsed_ms = (time.perf_counter() - start_time) * 1000

print(f"Optimized Forward Pass Duration: {opt_elapsed_ms:.2f} ms")
print(f"Latency Reduction: {((elapsed_ms - opt_elapsed_ms) / elapsed_ms) * 100:.1f}% faster")
print(f"Percentage of Tokens Dropped: {dropped_pct:.2f}%")

# Phase 3: Hardware Profiling & Expert Cache-Miss Simulation

### Objective
In true production environments (like serving massive Mixture of Experts models on consumer hardware or constrained clusters), you cannot fit all expert parameters into active GPU VRAM simultaneously.

### The Simulated Bottleneck
1. **Active VRAM Pool:** We simulate a constraint where only a subset of our total expert pool can sit in fast local GPU VRAM. The rest reside in slow system RAM.
2. **PCIe Transfer Tax (Cache Miss):** If the router assigns a token to an expert currently sitting in system RAM, the engine triggers an "Expert Cache Miss." We apply a strict latency penalty simulating the microsecond overhead of copying layer weights across a PCIe lane before execution can resume.

In [ ]:
import numpy as np

class HardwareMemorySimulator:
    def __init__(self, num_experts=8, max_vram_experts=4, pcie_latency_ms=45.0):
        self.num_experts = num_experts
        self.max_vram_experts = max_vram_experts
        self.pcie_latency_ms = pcie_latency_ms # Latency overhead per transfer block

        # Start with the first N experts pre-loaded into fast memory
        self.vram_cache = set(range(max_vram_experts))

    def simulate_inference_pass(self, routing_indices):
        """
        Processes token routing matrices and calculates hardware cache misses
        as experts are swapped dynamically across memory boundaries.
        """
        flat_indices = routing_indices.cpu().view(-1).numpy()
        unique_experts_requested = np.unique(flat_indices)

        cache_misses = 0
        simulated_transfer_overhead = 0.0

        # Track active hardware cache mutations
        for expert_id in unique_experts_requested:
            if expert_id not in self.vram_cache:
                cache_misses += 1
                simulated_transfer_overhead += self.pcie_latency_ms

                # Evict an expert (FIFO) and load the new one into VRAM
                evicted = list(self.vram_cache)[0]
                self.vram_cache.remove(evicted)
                self.vram_cache.add(expert_id)

        return cache_misses, simulated_transfer_overhead

# Extract actual routing indices from our Phase 2 optimized forward pass run
with torch.no_grad():
    _, indices, _ = optimized_moe.router(simulated_tokens.view(-1, D))

# Instantiate hardware simulator: 8 experts total, but only 4 can fit in active VRAM
gpu_memory_orchestrator = HardwareMemorySimulator(num_experts=8, max_vram_experts=4)
misses, structural_penalty = gpu_memory_orchestrator.simulate_inference_pass(indices)

print("--- RUNNING PHASE 3 HARDWARE PROFILING ---")
print(f"Total Expert Cache Misses Triggered: {misses}")
print(f"Simulated PCIe Hardware Stall Overhead: {structural_penalty:.2f} ms")
print(f"Total Projected Inference Time (Layer Execution + PCIe Tax): {opt_elapsed_ms + structural_penalty:.2f} ms")